In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
from brighteyes_flim import mcs
from brighteyes_flim import Alignment


In [ ]:
CHANNEL = 12
C_REF = 1.0
IRF_ITERATIONS = 300


def extract_channel_hist(data_6d, channel=CHANNEL):
    return np.asarray(data_6d[0, 0, :, :, :, channel].sum(axis=(0, 1)), dtype=float)


def build_time_axis(metadata):
    nbin = int(metadata.dfd_nbins)
    period_ns = 1 / (metadata.dfd_freq * 1e6) * 1e9
    dt_ns = period_ns / nbin
    t_ns = np.arange(nbin) * dt_ns
    return nbin, dt_ns, period_ns, t_ns


In [ ]:
f_ref = "/mnt/DATA/Mixed Data/26-03-17_Convallaria_and_FLIMLABS_calibrated_Yellow_Slide/FLIMLABS_Yellow_slide_2_5ns-17-03-2026-16-18-22.h5"
tau_ref_ns = 2.5

f_data = "/mnt/DATA/Mixed Data/26-03-17_Convallaria_and_FLIMLABS_calibrated_Yellow_Slide/01_Convallaria_DFD_40MHz-17-03-2026-16-59-41.h5"

ref_data_6d, metadata_ref = mcs.load(f_ref)
data_6d, metadata_data = mcs.load(f_data)

ref_hist_raw = extract_channel_hist(ref_data_6d)
data_hist_raw = extract_channel_hist(data_6d)

nbin, dt_ns, period_ns, t_ns = build_time_axis(metadata_data)

data_hist_raw_norm = data_hist_raw / data_hist_raw.sum()
ref_hist_raw_norm = ref_hist_raw / ref_hist_raw.sum()


In [ ]:
est_irf_ref = np.asarray(
    Alignment.IRF_from_data_deconvolution(
        ref_hist_raw_norm,
        t_ns,
        C_REF,
        tau_ref_ns,
        period_ns,
        iterations=IRF_ITERATIONS,
    ),
    dtype=float,
)
est_irf_ref_norm = est_irf_ref / est_irf_ref.sum()

setup_df = pd.DataFrame(
    [
        {
            "channel": CHANNEL,
            "nbin": nbin,
            "dt_ns": dt_ns,
            "period_ns": period_ns,
            "tau_ref_ns": tau_ref_ns,
            "reference_counts": ref_hist_raw.sum(),
            "data_counts": data_hist_raw.sum(),
        }
    ]
)

display(setup_df.round(6))


In [ ]:
fit_result, fit_cov = Alignment.perform_fit_data(
    t_ns,
    data_hist_raw_norm,
    est_irf_ref_norm,
    period_ns,
    fit_type="circular",
)

fitted_hist = np.asarray(
    Alignment.fit_model_data(
        t_ns,
        fit_result["C"],
        fit_result["dT"],
        fit_result["tau"],
        irf=est_irf_ref_norm,
        period=period_ns,
    ),
    dtype=float,
)

single_fit_df = pd.DataFrame(
    [
        {
            "C": float(fit_result["C"]),
            "dT_bins": float(fit_result["dT"]),
            "dT_ns": float(fit_result["dT"] * dt_ns),
            "tau_ns": float(fit_result["tau"]),
        }
    ]
)

display(single_fit_df.round(6))

plt.figure(figsize=(10, 4))
plt.plot(t_ns, ref_hist_raw_norm, label="Reference", alpha=0.5)
plt.plot(t_ns, est_irf_ref_norm, label="Estimated IRF")
plt.plot(t_ns, data_hist_raw_norm, label="Data")
plt.plot(t_ns, fitted_hist / fitted_hist.sum(), label="Circular fit")
plt.xlabel("Time (ns)")
plt.ylabel("Normalized intensity")
plt.title(f"Single fit: dT = {fit_result['dT'] * dt_ns:.3f} ns, tau = {fit_result['tau']:.3f} ns")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
shift_values = np.arange(nbin)
scan_rows = []

for shift in shift_values:
    ref_hist_shifted = np.roll(ref_hist_raw, shift)
    ref_hist_shifted_norm = ref_hist_shifted / ref_hist_shifted.sum()

    est_irf_shifted = np.asarray(
        Alignment.IRF_from_data_deconvolution(
            ref_hist_shifted_norm,
            t_ns,
            C_REF,
            tau_ref_ns,
            period_ns,
            iterations=IRF_ITERATIONS,
        ),
        dtype=float,
    )
    est_irf_shifted_norm = est_irf_shifted / est_irf_shifted.sum()

    fit_result_shifted, fit_cov_shifted = Alignment.perform_fit_data(
        t_ns,
        data_hist_raw_norm,
        est_irf_shifted_norm,
        period_ns,
        fit_type="circular",
    )

    scan_rows.append(
        {
            "shift": int(shift),
            "shift_ns": float(shift * dt_ns),
            "C": float(fit_result_shifted["C"]),
            "dT_bins": float(fit_result_shifted["dT"]),
            "dT_ns": float(fit_result_shifted["dT"] * dt_ns),
            "tau_ns": float(fit_result_shifted["tau"]),
            "irf_peak_bin": int(np.argmax(est_irf_shifted_norm)),
        }
    )

scan_results_df = pd.DataFrame(scan_rows)
scan_results_df["dT_bins_unwrapped"] = np.unwrap(scan_results_df["dT_bins"] / nbin * 2 * np.pi) / (2 * np.pi) * nbin
scan_results_df["dT_ns_unwrapped"] = scan_results_df["dT_bins_unwrapped"] * dt_ns

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(scan_results_df.round(6))

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(scan_results_df["shift"], scan_results_df["dT_ns_unwrapped"], color="tab:blue", marker=".")
axes[0].set_ylabel("Fit dT (ns, unwrapped)")
axes[0].set_title("Circular fit vs rolled reference shift")
axes[0].grid(True, alpha=0.3)

axes[1].plot(scan_results_df["shift"], scan_results_df["tau_ns"], color="tab:red", marker=".")
axes[1].set_xlabel("Reference roll shift (bins)")
axes[1].set_ylabel("Fit tau (ns)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
